# FraudShield: Exploratory Data Analysis & Time Analysis
**Project**: FraudShield – Explainable AI-Based Financial Fraud Detection & Risk Scoring
**Dataset**: Kaggle Credit Card Fraud Detection (284,807 transactions, 492 fraud cases)


In [ ]:
%matplotlib inline
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to path
sys.path.insert(0, os.path.abspath(".."))
from src.data.loader import load_raw_data, validate_dataset

sns.set_theme(style="darkgrid", palette="muted")
print("Environment successfully initialized!")


## 1. Dataset Loading and Schema Validation

In [ ]:
df = load_raw_data("../data/raw/creditcard.csv")
report = validate_dataset(df)
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Validation Passed: {report.validation_passed}")
print(f"Missing Values: {sum(report.missing_counts.values())}")
print(f"Duplicate Records: {report.duplicate_count:,}")
df.head()


## 2. Target Class Distribution & Imbalance

In [ ]:
class_counts = df["Class"].value_counts()
fraud_rate = (class_counts[1] / len(df)) * 100
print(f"Legitimate (Class 0): {class_counts[0]:,} ({100 - fraud_rate:.3f}%)")
print(f"Fraudulent (Class 1): {class_counts[1]:,} ({fraud_rate:.3f}%)")

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(x=["Legitimate (0)", "Fraud (1)"], y=class_counts.values, ax=ax, palette=["#4A90D9", "#dc3545"])
ax.set_yscale("log")
ax.set_title("Class Imbalance Distribution (Log Scale)", fontsize=14)
ax.set_ylabel("Transaction Count (Log Scale)")
for i, v in enumerate(class_counts.values):
    ax.text(i, v * 1.2, f"{v:,}", ha="center", fontweight="bold")
plt.show()


## 3. Transaction Amount Analysis

In [ ]:
print("=== Legitimate Transaction Amount Stats ===")
print(df[df["Class"] == 0]["Amount"].describe())

print("\n=== Fraudulent Transaction Amount Stats ===")
print(df[df["Class"] == 1]["Amount"].describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df[df["Class"] == 0]["Amount"], bins=50, kde=True, ax=axes[0], color="#4A90D9")
axes[0].set_title("Legitimate Transaction Amounts")
axes[0].set_xlim(0, 1000)

sns.histplot(df[df["Class"] == 1]["Amount"], bins=50, kde=True, ax=axes[1], color="#dc3545")
axes[1].set_title("Fraudulent Transaction Amounts")
axes[1].set_xlim(0, 1000)
plt.tight_layout()
plt.show()


## 4. Time-of-Day Transaction Patterns (Modulo 86400)

In [ ]:
# Derive approximate hour of day from elapsed seconds
df["hour_of_day"] = (df["Time"] % 86400) / 3600.0

fig, ax = plt.subplots(figsize=(12, 5))
sns.kdeplot(df[df["Class"] == 0]["hour_of_day"], label="Legitimate (Class 0)", color="#4A90D9", ax=ax, bw_adjust=0.5)
sns.kdeplot(df[df["Class"] == 1]["hour_of_day"], label="Fraudulent (Class 1)", color="#dc3545", ax=ax, bw_adjust=0.5)
ax.set_title("Transaction Density Across 24-Hour Day Cycle", fontsize=14)
ax.set_xlabel("Hour of Day (0 to 24)")
ax.set_ylabel("Kernel Density")
ax.legend()
plt.show()


## 5. Correlation Heatmap (Top Correlated Features)

In [ ]:
corrs = df.corr()["Class"].sort_values()
top_neg = corrs.head(6)
top_pos = corrs.tail(7).iloc[:-1] # exclude Class itself
top_corrs = pd.concat([top_neg, top_pos])

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(x=top_corrs.values, y=top_corrs.index, ax=ax, palette="vlag")
ax.set_title("Top Positively & Negatively Correlated Features with Fraud Target", fontsize=14)
ax.set_xlabel("Pearson Correlation with Class")
plt.show()
